# Hype Score Regression Tests

Test cases that validate algorithm behavior doesn't regress. Run this notebook after any algorithm changes.

## Test Philosophy

Each test case captures a real-world scenario where the algorithm should produce sensible rankings. The tests don't check exact scores (which will change with tuning) but validate **relative rankings** between games.

These are synthetic scoring fixtures, not observed Kalshi markets or historical backtests. Their fixed season, evaluation time, and source metadata exercise the same policy as the data loader.

In [ ]:
import sys
sys.path.insert(0, '../src')

from dataclasses import dataclass
from typing import Optional

from hype_score.config import HypeConfig
from hype_score.models import Game, PregameOdds, TeamRating
from hype_score.scoring import calculate_hype_score
from datetime import datetime, timedelta, timezone

evaluated_at = datetime(2026, 3, 15, 18, tzinfo=timezone.utc)

In [ ]:
@dataclass
class TestCase:
    name: str
    description: str
    higher_game: dict   
    lower_game: dict    

def make_game(sport: str, home: str, away: str, game_id: str = "test") -> Game:
    return Game(
        id=game_id,
        sport=sport,
        status="scheduled",
        home_team=home,
        away_team=away,
        home_team_abbreviation=home,
        away_team_abbreviation=away,
        scheduled_start=evaluated_at + timedelta(hours=2),
        season_year=2026,
    )

def make_odds(home_win_prob: float, game_id: str = "test") -> PregameOdds:
    return PregameOdds(
        game_id=game_id,
        home_win_probability=home_win_prob,
        away_win_probability=1 - home_win_prob,
    )

def make_rating(
    sport: str, 
    abbr: str, 
    bpi: Optional[float] = None,
    playoff_prob: float = 0.5,
    champ_prob: float = 0.01
) -> TeamRating:
    return TeamRating(
        sport=sport,
        team_abbreviation=abbr,
        season=2026,
        source="kalshi",
        source_season=2026,
        fetched_at=evaluated_at,
        bpi_rating=bpi,
        playoff_probability=playoff_prob,
        championship_probability=champ_prob,
    )

def run_test(test: TestCase, config: HypeConfig = None) -> tuple[bool, str]:
    if config is None:
        config = HypeConfig.default()
    
    h = test.higher_game
    higher_result = calculate_hype_score(
        make_game(h['sport'], h['home'], h['away'], 'higher'),
        make_odds(h['home_win_prob'], 'higher'),
        make_rating(h['sport'], h['home'], h.get('home_bpi'), h.get('home_playoff', 0.5), h.get('home_champ', 0.01)),
        make_rating(h['sport'], h['away'], h.get('away_bpi'), h.get('away_playoff', 0.5), h.get('away_champ', 0.01)),
        config,
        evaluated_at=evaluated_at,
    )
    
    l = test.lower_game
    lower_result = calculate_hype_score(
        make_game(l['sport'], l['home'], l['away'], 'lower'),
        make_odds(l['home_win_prob'], 'lower'),
        make_rating(l['sport'], l['home'], l.get('home_bpi'), l.get('home_playoff', 0.5), l.get('home_champ', 0.01)),
        make_rating(l['sport'], l['away'], l.get('away_bpi'), l.get('away_playoff', 0.5), l.get('away_champ', 0.01)),
        config,
        evaluated_at=evaluated_at,
    )
    
    passed = higher_result.hype_score > lower_result.hype_score
    
    msg = f"""
    Higher game ({h['home']} vs {h['away']}): {higher_result.hype_score:.1f}
      - Stakes: {higher_result.components.stakes_score:.1f}
      - Quality: {higher_result.components.quality_score:.1f}
      - Closeness: {higher_result.components.closeness_score:.1f}
    Lower game ({l['home']} vs {l['away']}): {lower_result.hype_score:.1f}
      - Stakes: {lower_result.components.stakes_score:.1f}
      - Quality: {lower_result.components.quality_score:.1f}
      - Closeness: {lower_result.components.closeness_score:.1f}
    Difference: {higher_result.hype_score - lower_result.hype_score:+.1f}
    """
    
    return passed, msg

## Test Case 1: Contenders vs Bottom-Feeders (Jan 2026)

**Issue**: BKN @ WSH (two bad teams, close game) ranked higher than DEN @ CLE (two good teams, lopsided odds).

**Expected behavior**: Games between two playoff contenders should rank higher than games between two eliminated teams, even if the contender game has worse odds spread.

In [ ]:
test_contenders_vs_bottomfeeders = TestCase(
    name="Contenders vs Bottom-Feeders",
    description="Two playoff contenders should rank higher than two eliminated teams",
    higher_game={
        'sport': 'nba',
        'home': 'CLE',
        'away': 'DEN',
        'home_win_prob': 0.858,   
        'home_bpi': 4.17,
        'away_bpi': 4.31,
        'home_playoff': 0.94,
        'away_playoff': 0.995,
        'home_champ': 0.01,
        'away_champ': 0.08,
    },
    lower_game={
        'sport': 'nba',
        'home': 'WSH',
        'away': 'BKN',
        'home_win_prob': 0.593,   
        'home_bpi': -8.84,
        'away_bpi': -4.22,
        'home_playoff': 0.005,
        'away_playoff': 0.005,
        'home_champ': 0.005,
        'away_champ': 0.005,
    },
)

passed, msg = run_test(test_contenders_vs_bottomfeeders)
print(f"{'PASS' if passed else 'FAIL'}: {test_contenders_vs_bottomfeeders.name}")
print(msg)

## Test Case 2: Close Game Between Good Teams

**Expected behavior**: A close game between two good teams should rank very high.

In [ ]:
test_close_good_teams = TestCase(
    name="Close Good Teams vs Close Bad Teams",
    description="A close game between good teams should outrank close game between bad teams",
    higher_game={
        'sport': 'nba',
        'home': 'BOS',
        'away': 'MIL',
        'home_win_prob': 0.55,   
        'home_bpi': 8.0,
        'away_bpi': 6.0,
        'home_playoff': 0.99,
        'away_playoff': 0.95,
        'home_champ': 0.15,
        'away_champ': 0.05,
    },
    lower_game={
        'sport': 'nba',
        'home': 'WSH',
        'away': 'BKN',
        'home_win_prob': 0.50,   
        'home_bpi': -8.84,
        'away_bpi': -4.22,
        'home_playoff': 0.005,
        'away_playoff': 0.005,
        'home_champ': 0.005,
        'away_champ': 0.005,
    },
)

passed, msg = run_test(test_close_good_teams)
print(f"{'PASS' if passed else 'FAIL'}: {test_close_good_teams.name}")
print(msg)

## Test Case 3: Bubble Teams Have Higher Stakes Than Contenders

**Expected behavior**: While contenders get a bonus, teams actually ON the playoff bubble (50% playoff odds) should still have meaningful stakes.

In [ ]:
test_bubble_relevance = TestCase(
    name="Bubble Teams Still Matter",
    description="A close game between bubble teams should outrank lopsided contender game",
    higher_game={
        'sport': 'nba',
        'home': 'ATL',
        'away': 'CHI',
        'home_win_prob': 0.52,   
        'home_bpi': 2.0,
        'away_bpi': 1.5,
        'home_playoff': 0.50,   
        'away_playoff': 0.45,   
        'home_champ': 0.01,
        'away_champ': 0.01,
    },
    lower_game={
        'sport': 'nba',
        'home': 'CLE',
        'away': 'DEN',
        'home_win_prob': 0.858,   
        'home_bpi': 4.17,
        'away_bpi': 4.31,
        'home_playoff': 0.94,
        'away_playoff': 0.995,
        'home_champ': 0.01,
        'away_champ': 0.08,
    },
)

passed, msg = run_test(test_bubble_relevance)
print(f"{'PASS' if passed else 'FAIL'}: {test_bubble_relevance.name}")
print(msg)

## Run All Tests

In [ ]:
all_tests = [
    test_contenders_vs_bottomfeeders,
    test_close_good_teams,
    test_bubble_relevance,
]

print("="*60)
print("REGRESSION TEST RESULTS")
print("="*60)

all_passed = True
for test in all_tests:
    passed, _ = run_test(test)
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {test.name}")
    if not passed:
        all_passed = False

print("="*60)
if all_passed:
    print("All tests passed!")
else:
    print("SOME TESTS FAILED - Review algorithm changes")